In [ ]:
# /// script
# requires-python = ">=3.14"
# dependencies = [
#     "jupytergis==0.16.8",
#     "openeo",
# ]
# ///

## Dependencies

Install [`uv`](https://docs.astral.sh/uv) and then launch this notebook with:

```
uvx juv run notebooks/flood_mapping_sentinel1.ipynb
```

(The `uvx` command is included when installing `uv`).

## License

This notebook is an adaptation of the openEO-UDP notebook
[flood_mapping.ipynb](https://github.com/developmentseed/openeo-udp/blob/main/notebooks/sentinel/sentinel-1/water_and_flood_monitoring/flood_mapping.ipynb)
by Development Seed. That notebook is a conversion of a Sentinel Hub evalscript and is licensed under **CC-BY-SA-4.0**.
This adaptation is also licensed under **CC-BY-SA-4.0**.

- Original evalscript: https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-1/flood_mapping/ (Maryam Salehi)
- Source: Sentinel Hub Custom Scripts (CC-BY-SA-4.0)
- Conversion: Development Seed (openEO-UDP project)

# Flood mapping of the Chenab river flood, Punjab, Pakistan, September 2025

This notebook maps the monsoon flood of the Chenab river around Jhang, Punjab, Pakistan, in late August and early September 2025, from Sentinel-1 radar data with openEO.
At the Trimmu headworks, where the Jhelum joins the Chenab, the river reached a "very high" flood on 1 and 2 September 2025.
It shows the result as a dynamic tile layer in a JupyterGIS map, on top of an OpenStreetMap basemap.

Radar sees through clouds, so it can map a flood during the monsoon.
Calm open water reflects the radar pulse away from the sensor, so flooded land becomes dark.
The algorithm compares one VV acquisition from *before* the flood with one from *during* the flood, and shows the pair as one RGB image.

The steps are:
1. Connect to the openEO backend.
2. Load the Sentinel-1 GRD VV acquisitions from 20 August 2025 (before) and 1 September 2025 (during), and calibrate them to backscatter with `sar_backscatter`.
3. Merge the two dates into one data cube with the bands `vv_before` and `vv_during`.
4. Build the change composite `[1.5 × VV_before, 1.5 × VV_during, 1.5 × VV_during]`.
5. Show the result in JupyterGIS.

## Connect to the openEO backend

This notebook uses the openEO backend at `https://openeo.ds.io/`. You log in with a [Copernicus Data Space Ecosystem](https://dataspace.copernicus.eu/) account.

In [ ]:
import openeo
from openeo.processes import array_create

connection = openeo.connect("https://openeo.ds.io/")

In [ ]:
connection.authenticate_oidc()

## Parameters

The bounding box covers the Chenab floodplain around the city of Jhang and the Trimmu headworks.

Each time range is one day, so it holds a single Sentinel-1 pass (two adjacent scenes of the same pass cover the area). The end date is exclusive in openEO.
The two dates are 12 days apart and come from the **same relative orbit (173, ascending)**.
This is important: backscatter depends on the viewing geometry, so two dates from different orbits show large differences that have nothing to do with water.

The Sentinel-1 catalogue of this backend is complete only from December 2024, so pick dates after that for other floods.

`gain` is the display gain of the original script. It scales the linear backscatter (roughly 0.05 to 0.4 over land) to the 0 to 1 range.

In [ ]:
location_name = "Jhang, Punjab, Pakistan"

bounding_box = {"west": 72.0, "south": 30.95, "east": 72.55, "north": 31.45}
time_before = ["2025-08-20", "2025-08-21"]  # before the flood
time_during = ["2025-09-01", "2025-09-02"]  # during the Chenab flood peak
gain = 1.5

## Load and calibrate the Sentinel-1 data

The `sentinel-1-grd` collection has the raw Level-1 GRD digital numbers.
`sar_backscatter` calibrates them to linear backscatter with the calibration and thermal noise tables of each scene.
The backend supports only the ellipsoid coefficients, so we use `gamma0-ellipsoid`, the default of the Sentinel Hub `S1GRD` data that the original script uses.
Ellipsoid backscatter has no terrain correction, but this is good for flood mapping over flat terrain, where both dates have the same geometry.

`sar_backscatter` must come directly after `load_collection`.
`firstpixel` then makes a mosaic of the time range, in case the area covers two scenes of the same pass.

Both cubes have a band called `vv`, so we rename the bands to `vv_before` and `vv_during` with `rename_labels` before we merge them.

In [ ]:
def load_vv(temporal_extent, label):
    """Load one Sentinel-1 VV acquisition as calibrated linear backscatter."""
    cube = connection.load_collection(
        "sentinel-1-grd",
        spatial_extent=bounding_box,
        temporal_extent=temporal_extent,
        bands=["vv"],
    )
    cube = cube.sar_backscatter(coefficient="gamma0-ellipsoid")
    cube = cube.reduce_dimension(dimension="t", reducer="firstpixel")
    return cube.rename_labels(
        dimension=cube.metadata.band_dimension.name, target=[label]
    )


vv_before = load_vv(time_before, "vv_before")
vv_during = load_vv(time_during, "vv_during")

flood_pair = vv_before.merge_cubes(vv_during)
flood_pair.metadata.band_names

## Flood change composite

The composite puts the image from before the flood in the red channel, and the image during the flood in the green and blue channels:

| Channel | Value |
|---|---|
| Red | 1.5 × VV (before) |
| Green | 1.5 × VV (during) |
| Blue | 1.5 × VV (during) |

Green and blue are the same, so the colors are on a red to cyan axis:

| Color | Meaning |
|---|---|
| Red | Bright before, dark during: **newly flooded land** |
| Cyan | Dark before, bright during: double bounce between water and buildings or vegetation (flooded urban areas) |
| Black | Dark in both: permanent water (river channels, canals) |
| Grey / white | Similar in both: land without change (bright white is urban) |

In [ ]:
def flood_composite(data):
    """
    Bi-temporal Sentinel-1 flood mapping.
    Input data array: [vv_before, vv_during]

    Returns [1.5 * VV_before, 1.5 * VV_during, 1.5 * VV_during].
    """
    before = gain * data[0]
    during = gain * data[1]
    return array_create([before, during, during])


# Apply the composite on the data cube on the bands dimension
flood_image = flood_pair.apply_dimension(
    dimension=flood_pair.metadata.band_dimension.name, process=flood_composite
)

# Linear scale to 0-255 for RGB visualization
flood_image = flood_image.linear_scale_range(
    input_min=0, input_max=1, output_min=0, output_max=255
)

graph = flood_image

## Show the result in JupyterGIS

The map has an OpenStreetMap basemap. The openEO layer is added on top of it.
`add_openeo_tile_layer` creates an XYZ service on the backend and fits the map to the bounding box.

The backend calculates each tile on request. A tile can take 10 to 25 seconds, so the layer fills in slowly.

Red is newly flooded land, black is permanent water, cyan is flooded urban areas and grey is land without change.

In [ ]:
from jupytergis import GISDocument

doc = GISDocument(latitude=31.2, longitude=72.275, zoom=10)

doc.add_raster_layer(
    url="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    name="OpenStreetMap",
    attribution="© OpenStreetMap contributors",
)

In [ ]:
doc.add_openeo_tile_layer(graph, name="Sentinel-1 Flood Mapping")
doc

## Technical notes

**Applications:**
- Rapid flood mapping during a storm or the monsoon, when clouds block optical satellites.
- Damage assessment and emergency response.
- Mapping how far a flood reaches on agricultural plains and in towns.

**Limitations:**
- The two dates must come from the same relative orbit and pass direction. The process graph cannot check this.
- Seasonal water looks the same as flood water. In Punjab, rice paddies are flooded on a schedule during the monsoon, so part of the red area can be irrigated fields, not flood damage.
- Smooth dry surfaces (tarmac, dry sand, harvested fields) and radar shadow behind steep terrain also get dark without a flood. Wind on flood water makes it rougher and brighter, so it can be missed.
- `gamma0-ellipsoid` has no terrain correction. It is good for flat terrain such as the Punjab plain, but not for mountains.
- There is no speckle filter, so the image has some salt-and-pepper noise.

**Binary flood mask:** the original script also has a binary mask: a pixel is flooded when `1.5 × (VV_before − VV_during) > 0.05` (or `0.08` for a less sensitive mask). Flooded urban areas get brighter, not darker, so the mask does not include them.

## Attribution

**Original script:** [Flood Mapping With Sentinel-1](https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-1/flood_mapping/) by Maryam Salehi, [Sentinel Hub Custom Scripts](https://github.com/sentinel-hub/custom-scripts)

**Methodology:**
- Martinis, S., Kersten, J., Twele, A. (2015). A fully automated TerraSAR-X based flood service. *ISPRS Journal of Photogrammetry and Remote Sensing*, 104, 203–212.
- Shen, X. et al. (2019). Near-real-time non-obstructed flood inundation mapping using synthetic aperture radar. *Remote Sensing of Environment*, 221, 302–315.